# 

In [6]:
import awkward as ak
import vector
vector.register_awkward()
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import hist
import mplhep as hep
import pandas as pd
hep.style.ROOT
import matplotlib.cm as cm
import sys
sys.path.insert(0, '../scripts')
from plotting import *
from common_defs import *
print(matplotlib.__version__)

%load_ext autoreload
%autoreload 2

FileNotFoundError: [Errno 2] No such file or directory: '/oscar/data/mleblan6/rjain/xmd_outputs/jeppe_XMD.npy'

In [ ]:
zjj_jets = ak.from_parquet(lhayDir + '/zjj_NLO_100k_alljets.parquet')
order = ak.argsort(zjj_jets.pt, axis=1, ascending=False)
zjets = zjj_jets[order]
####load hp points to repopulate weights where the event==0 and those weights were not computed
hp_points = np.load(rjainDir + "/ppzjj_100k/hardprocess_points.npy")
####Load z coords
zpt = np.load(rjainDir + '/ppzjj_100k/ppzjj_NLO_100k_zcoords.npy')[:,0]
zeta = np.load(rjainDir + '/ppzjj_100k/ppzjj_NLO_100k_zcoords.npy')[:,1]
zphi = np.load(rjainDir + '/ppzjj_100k/ppzjj_NLO_100k_zcoords.npy')[:,2]


#### Here lies plots for Z_jet events. Edit styles in plotting.py.

In [ ]:
obsDict = {}
njets = max(ak.num(zjets.pt, axis=1))

nosel = (ak.num(zjets.pt, axis=1)>-5)

sel_twojets = (ak.num(zjets.pt, axis=1)>1)
sel_jets_twojets = zjets[sel_twojets]

sel_onejet = (ak.num(zjets.pt, axis=1)>0)
sel_jets_onejet = zjets[sel_onejet]

obsDict["drjj"] = {"name": r"\Delta R_{j_0, j_1}", "units": "", "xmin": 0, "xmax": 10, "numbins": 20, "ratioLim": [0.45,1.55], "obs": np.sqrt((sel_jets_twojets.eta[:,0] - sel_jets_twojets.eta[:,1])**2 + (sel_jets_twojets.phi[:,0] - sel_jets_twojets.phi[:,1])**2), "sel":sel_twojets, "logx":False, "ymin":2e-5, "ymax": 5e0, "logy": True}
obsDict["drzj"] = {"name": r"\Delta R_{Z, j_0}","units": "", "xmin": 0, "xmax": 6.5, "numbins": 20, "ratioLim": [0.7, 1.3], "obs": np.sqrt((zeta[sel_onejet] - sel_jets_onejet.eta[:,0])**2 + (zphi[sel_onejet] - sel_jets_onejet.phi[:,0])**2), "sel":sel_onejet, "logx":False, "ymin":6e-3, "ymax": 2e0, "logy": True}
obsDict["njets"] = {"name": r'N_{\mathrm{jets}}',"units": "", "xmin": 0, "xmax": njets, "numbins": njets, "ratioLim": [0.7, 1.3], "obs": ak.num(zjets.pt, axis=1), "sel":nosel, "logx":False, "ymin":2e-5, "ymax": 6e0, "logy": True}
obsDict["ptlead"] = {"name": r'p_{\mathrm{T},j_0}',"units": r"\mathrm{[GeV]}", "xmin": 20, "xmax": 150, "numbins": 20, "ratioLim": [0.45,1.55], "obs": sel_jets_onejet.pt[:,0], "sel": sel_onejet, "logx":False, "ymin":3e-3, "ymax": 8e-1, "logy": True}
obsDict["ht"] = {"name": r'H_\mathrm{T}',"units": r"\mathrm{[GeV]}", "xmin": 50, "xmax": 700, "numbins": 20, "ratioLim": [0.7,1.3], "obs": ak.sum(zjets.pt, axis=-1), "sel": nosel, "logx":True, "ymin":3e-3, "ymax": 5e-1, "logy": True}
obsDict["zpt"] = {"name": r'p_{\mathrm{T},Z}',"units": r"\mathrm{[GeV]}", "xmin": 10, "xmax": 350, "numbins": 20, "ratioLim": [0.7,1.3], "obs": zpt, "sel": nosel, "logx":True, "ymin":1e-3, "ymax": 5e-1, "logy": True}
obsDict["ptrat"] = {"name": r"p_{\mathrm{T}0}/p_{\mathrm{T}1}","units": "", "xmin": 1, "xmax": 4.0, "numbins": 14, "ratioLim": [0.45,1.55], "obs": sel_jets_twojets.pt[:,0]/sel_jets_twojets.pt[:,1], "sel": sel_twojets, "logx":False, "ymin":1.1e-4, "ymax": 2e0, "logy": True}

In [ ]:
processDict = {}
processDict["Zjets"] = {"name": "Z+jets", "weights": weights_orig_z}

In [ ]:
comparisonDict = {}

comparisonDict["stages"] = {"weights": ["zjet_hp_df", "zjet_ps_df", "zjet_had_df"], "maxCellRadius": 30}
comparisonDict["betasHad"] = {"weights": ["zjet_b0_had_df", "zjet_b0p5_had_df", "zjet_had_df", "zjet_b2_had_df",  "zjet_binf_had_df"], "maxCellRadius": 250}
comparisonDict["betasHp"] = {"weights": ["zjet_b0_hp_df", "zjet_b0p5_hp_df", "zjet_hp_df", "zjet_b2_hp_df",  "zjet_binf_hp_df"], "maxCellRadius": 250}
comparisonDict["semdVemd_Had"] = {"weights": ["zjet_SEMD_had_df", "zjet_had_df", "zjet_jeppe_df"], "maxCellRadius": 50}

comparisonDict["stagesBeta0"] = {"weights": ["zjet_b0_hp_df", "zjet_b0_ps_df", "zjet_b0_had_df"], "maxCellRadius": 30}
comparisonDict["stagesBetap5"] = {"weights": ["zjet_b0p5_ps_df", "zjet_b0p5_hp_df", "zjet_b0p5_had_df"], "maxCellRadius": 100}
comparisonDict["stagesBeta2"] = {"weights": ["zjet_b2_hp_df", "zjet_b2_ps_df", "zjet_b2_had_df"], "maxCellRadius": 3}

comparisonDict["betasHP"] = {"weights": ["zjet_b0_hp_df", "zjet_b0p5_hp_df", "zjet_hp_df", "zjet_b2_hp_df",  "zjet_binf_hp_df"], "maxCellRadius": 250}
comparisonDict["betasPS"] = {"weights": ["zjet_b0_ps_df", "zjet_b0p5_ps_df", "zjet_ps_df", "zjet_b2_ps_df",  "zjet_binf_ps_df"], "maxCellRadius": 250}

#comparisonDict["semdVemd_HP"] = {"weights": ["zjet_SEMD_hp_df", "zjet_hp_df", "zjet_SEMD_cm"], "maxCellRadius": 50}
#comparisonDict["semdVemd_PS"] = {"weights": ["zjet_SEMD_ps_df", "zjet_ps_df"], "maxCellRadius": 50}

#comparisonDict["semdStages"] = {"weights": ["zjet_SEMD_hp_df", "zjet_SEMD_ps_df", "zjet_SEMD_had_df", "zjet_SEMD_cm"], "maxCellRadius": 250}
#comparisonDict["semdp1Stages"] = {"weights": ["zjet_SEMD_hp_df", "zjet_SEMD_ps_df", "zjet_SEMD_had_df"], "maxCellRadius": 0.02}



In [ ]:
from plotting import *
for comparison in comparisonDict:
    rw_dict_strings = comparisonDict[comparison]["weights"]
    datasets = []
    for sample in rw_dict_strings:
        datasets.append(dataFrames[sample])
            
    directory = f"../plots/radius"
    plot_radii(datasets, directory, comparison, comparisonDict)
    

In [ ]:
rwFracs = [0.25, 0.75]
for comparison in comparisonDict:
    for observable in obsDict:
        for rwFrac in rwFracs:
            rw_dict_strings = comparisonDict[comparison]["weights"]

            processes = []
            datasets = []
            for sample in rw_dict_strings:
                datasets.append(dataFrames[sample])
                processes.append(dataFrames[sample]["process"])

            processWeights = processDict[processes[0]]["weights"]
            processTitle = processDict[processes[0]]["name"]
            isSameProcess = True
            for process in processes:
                if process != processes[0]:
                    print("Mixed processes -- this script requires rewriting to work in this context")
                    isSameProcess= False
                    break
            if not isSameProcess:
                break


            plot_diff_rw(obsDict[observable], datasets, processWeights, process_title = processTitle, title = comparison, channel = processes[0], rwFrac=rwFrac, obsName = observable)

       

In [ ]:
from common_defs import *
for comparison in comparisonDict:
    rw_dict_strings = comparisonDict[comparison]["weights"]
    neg_percents = []
    datasets = []
    variances = []
    for sample in rw_dict_strings:
        datasets.append(dataFrames[sample])
        neg_percent = get_negative_percent(processWeights, dataFrames[sample]["df"]["weights"])
        variance = [np.var(np.asarray(i)/14550) for i in dataFrames[sample]["df"]["weights"]]
        orig_variance = np.var(processWeights)
        if(neg_percent[0] < 1e-3):
            neg_percent = neg_percent[1:]
            variance = variance[1:]
        variances.append(variance/orig_variance)
        neg_percents.append(neg_percent)

        
    plot_variance(neg_percents, variances, datasets, title=comparison)


In [ ]:
from plotting import *
for comparison in comparisonDict:
    rw_dict_strings = comparisonDict[comparison]["weights"]

    neg_percents = []
    dilutions = []
    datasets = []
    for sample in rw_dict_strings:
        datasets.append(dataFrames[sample])
        neg_percent = get_negative_percent(processDict[dataFrames[sample]["process"]]["weights"], dataFrames[sample]["df"]["weights"])
        dilution = get_dilution(dataFrames[sample]["df"]["weights"])
        if(neg_percent[0] < 1e-3):
            neg_percent = neg_percent[1:]
            dilution = dilution[1:]
        neg_percents.append(neg_percent)
        dilutions.append(dilution)
        processTitle = processDict[dataFrames[sample]["process"]]["name"]
        
    plot_dilution(neg_percents, dilutions, datasets, title=comparison, process_title = processTitle)



In [ ]:
 for dataFrame in dataFrames:
     for observable in obsDict:
         xmin = obsDict[observable]["xmin"]
         xmax = obsDict[observable]["xmax"]
         ymin = obsDict[observable]["ymin"]
         ymax = obsDict[observable]["ymax"]
         numbins = obsDict[observable]["numbins"]
         obs_str = obsDict[observable]["name"]
         obs= obsDict[observable]["obs"]
         sel = obsDict[observable]["sel"]
         ratioLim = obsDict[observable]["ratioLim"]
         title = dataFrames[dataFrame]["title"]
         units = obsDict[observable]["units"]

         process_str = processDict[process]["name"]
         processWeights = processDict[process]["weights"]

         plot_same_rw_all(obs, dataFrames[dataFrame]["df"], processWeights, numbins, xmin, xmax, sel=sel, obs_str=obs_str, obs_title = observable, title = title, process_title=process_str, units=units, ymin=ymin, ymax=ymax, raxlim=ratioLim)


In [ ]:
jets_highstat = ak.from_parquet('/oscar/data/mleblan6/rjain/ppzjj_10M/ppzjj_NLO_10M_alljet.parquet')
order = ak.argsort(jets_highstat.pt, axis=1, ascending=False)
zjets_hs = jets_highstat[order]

####Load z coords
zpt = np.load('/oscar/data/mleblan6/rjain/ppzjj_10M/ppzjj_NLO_10M_zcoords.npy')[:,0]
zeta = np.load('/oscar/data/mleblan6/rjain/ppzjj_10M/ppzjj_NLO_10M_zcoords.npy')[:,1]
zphi = np.load('/oscar/data/mleblan6/rjain/ppzjj_10M/ppzjj_NLO_10M_zcoords.npy')[:,2]


weights_orig_highstat = np.load("/oscar/data/mleblan6/rjain/ppzjj_10M/ppzjj_NLO_10M_weight.npy")


obsDict_highstat = {}

njets = max(ak.num(zjets_hs.pt, axis=1))

nosel = (ak.num(zjets_hs.pt, axis=1)>-5)

sel_twojets = (ak.num(zjets_hs.pt, axis=1)>1)
sel_jets_twojets = zjets_hs[sel_twojets]

sel_onejet = (ak.num(zjets_hs.pt, axis=1)>0)
sel_jets_onejet = zjets_hs[sel_onejet]

obsDict_highstat["drjj"] = {"name": r"\Delta R_{j_0, j_1}", "units": "", "xmin": 0, "xmax": 10, "numbins": 20, "ratioLim": [0.45,1.55], "obs": np.sqrt((sel_jets_twojets.eta[:,0] - sel_jets_twojets.eta[:,1])**2 + (sel_jets_twojets.phi[:,0] - sel_jets_twojets.phi[:,1])**2), "sel":sel_twojets, "logx":False, "ymin":2e-5, "ymax": 5e0}
obsDict_highstat["drzj"] = {"name": r"\Delta R_{Z, j_0}","units": "", "xmin": 0, "xmax": 6.5, "numbins": 20, "ratioLim": [0.8, 1.2], "obs": np.sqrt((zeta[sel_onejet] - sel_jets_onejet.eta[:,0])**2 + (zphi[sel_onejet] - sel_jets_onejet.phi[:,0])**2), "sel":sel_onejet, "logx":False, "ymin":6e-3, "ymax": 2e0}
obsDict_highstat["njets"] = {"name": r'N_{\mathrm{jets}}',"units": "", "xmin": 0, "xmax": njets, "numbins": njets, "ratioLim": [0.5, 1.5], "obs": ak.num(zjets_hs.pt, axis=1), "sel":nosel, "logx":False, "ymin":2e-5, "ymax": 6e0}
obsDict_highstat["ptlead"] = {"name": r'p_{\mathrm{T},j_0}',"units": r"\mathrm{[GeV]}", "xmin": 20, "xmax": 150, "numbins": 20, "ratioLim": [0.45,1.55], "obs": sel_jets_onejet.pt[:,0], "sel": sel_onejet, "logx":False, "ymin":3e-3, "ymax": 8e-1}
obsDict_highstat["ht"] = {"name": r'H_\mathrm{T}',"units": r"\mathrm{[GeV]}", "xmin": 50, "xmax": 700, "numbins": 20, "ratioLim": [0.75,1.25], "obs": ak.sum(zjets.pt, axis=-1), "sel": nosel, "logx":True, "ymin":3e-3, "ymax": 5e-1}
obsDict_highstat["zpt"] = {"name": r'p_{\mathrm{T},Z}',"units": r"\mathrm{[GeV]}", "xmin": 10, "xmax": 350, "numbins": 20, "ratioLim": [0.75,1.25], "obs": zpt, "sel": nosel, "logx":True, "ymin":1e-3, "ymax": 5e-1}
obsDict_highstat["ptrat"] = {"name": r"p_{\mathrm{T}0}/p_{\mathrm{T}1}","units": "", "xmin": 1, "xmax": 4.0, "numbins": 14, "ratioLim": [0.45,1.55], "obs": sel_jets_twojets.pt[:,0]/sel_jets_twojets.pt[:,1], "sel": sel_twojets, "logx":False, "ymin":1.1e-4, "ymax": 2e0}

In [ ]:
rwFracs = [0.25, 0.5, 0.75]
observable = "njets"
logy=True
weights = [weights_orig_z, weights_orig_highstat]
logx = obsDict[observable]["logx"]
xmin = obsDict[observable]["xmin"]
xmax = obsDict[observable]["xmax"]
ymin = obsDict[observable]["ymin"]
ymax = obsDict[observable]["ymax"]
nbins = obsDict[observable]["numbins"]
obs_str = obsDict[observable]["name"]
obs = obsDict[observable]["obs"]
obs_hs = obsDict_highstat[observable]["obs"]
sel = np.array(obsDict[observable]["sel"])
sel_hs = np.array(obsDict_highstat[observable]["sel"])
ratioLim = obsDict[observable]["ratioLim"]
units = obsDict[observable]["units"]

jets_other = ak.from_parquet("/users/lhay/NegativeWeights/zjj_NLO_100k_alljetsNOLEP.parquet")
obs_other = ak.num(jets_other.pt, axis=1)


if logx:
    bins = np.logspace(np.log10(xmin), np.log10(xmax), nbins)
    axis_o = hist.axis.Variable(bins,name="data",label="orig",)
    axis_rw = hist.axis.Variable(bins,name="data",label="reweighted",)
else:
    axis_o = hist.axis.Regular(nbins,xmin, xmax,name="data",label="orig",)
    axis_rw = hist.axis.Regular(nbins,xmin, xmax,name="data",label="orig",)
if sel is None:
    sel  = np.ones_like(weights_orig_highstat, dtype=bool)
fig, ax = plt.subplots(nrows=1,
                    ncols=1,
                    figsize=(8,8),)
h_orig = hist.Hist(
    axis_o,
    storage=hist.storage.Weight(), )
h_orig.fill(obs, weight = weights_orig_z)
h_orig = h_orig/h_orig.sum(flow=False).value
hep.histplot(h_orig, ax=ax, label = "100K Original", color='blue')
h_orig_hs = hist.Hist(
    axis_o,
    storage=hist.storage.Weight(), )
h_orig_hs.fill(obs_hs, weight = weights_orig_highstat)
h_orig_hs = h_orig_hs/h_orig_hs.sum(flow=False).value
hep.histplot(h_orig_hs, ax=ax, label = "10M Original", color='black')
h_orig2 = hist.Hist(
    axis_o,
    storage=hist.storage.Weight(), )
h_orig2.fill(obs_other, weight = weights_orig_z)
h_orig2 = h_orig2/h_orig2.sum(flow=False).value
hep.histplot(h_orig2, ax=ax, label = "100k nolep", color='red')
ax.legend(frameon=False, fontsize=12, loc="upper right", borderpad=1.0)
ax.set_xlabel(r"$N_{jets}$")